# Crosswind 1-cosine gust response example

このNotebookでは、`SampleGlider.stab`の線形空力モデルを使い、横方向の1-cosine gustに対して次を順番に確認する。

1. 1-cosine gustの波形と外乱横滑り角
2. 線形4状態モデルの行列 $\mathbf{A}$ と突風入力ベクトル $\mathbf{B}_g$
3. Taylor係数 $K_1,K_2,K_3$
4. 簡易横風突風ロール指数
5. 6自由度時刻歴とピーク応答指標

横風突風の符号規約は次で統一する。

$$
U_{ds}>0
\quad:\quad
\text{航空機の右側から左側へ吹く横風突風}
$$

公開時刻歴では、

- `gust_velocity_y > 0`：右から吹く横風突風
- `airmass_velocity_y < 0`：機体軸$+y_b$右方に対して左向きの大気速度
- `beta_g > 0`：右からの突風による正の外乱横滑り角
- `beta_body`：機体の剛体速度から求めた横滑り角
- `beta`：相対風から求めた有効対気横滑り角

となる。

ここで使う質量・慣性モーメントは`SampleGlider`用の数値例であり、G103Aの実機MassProp値ではない。

## 1. importとリポジトリ位置

In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

start_dir = Path.cwd().resolve()
repo_root = next(
    (
        candidate
        for candidate in [start_dir, *start_dir.parents]
        if (candidate / "src" / "RollRudderGain.py").exists()
    ),
    None,
)

if repo_root is None:
    raise FileNotFoundError(
        "Could not find the repository root containing "
        "src/RollRudderGain.py."
    )

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.RollRudderGain import (  # noqa: E402
    calculate_crosswind_gust_response_indices,
    one_cosine_gust_velocity,
    plot_6dof_history,
    simulate_6dof_crosswind_gust_from_stab,
    write_6dof_history_csv,
)
from src.VSPAEROStab import read_vspaero_stab  # noqa: E402
from src.VvGammaChart import calculate_stab_basic_indices  # noqa: E402

print("repo_root:", repo_root)

## 2. 入力モデルと解析条件

定式化記事の比較条件と同じ値を使用する。

`Uds=3.0`は右から吹くピーク突風速度、`H=9.144 m`は30 ftの突風勾配距離である。

In [ ]:
stab_path = (
    repo_root
    / "examples"
    / "models"
    / "SampleGlider"
    / "SampleGlider.stab"
)
output_dir = (
    repo_root
    / "examples"
    / "notebooks"
    / "results"
    / "crosswind_gust_response"
)
output_dir.mkdir(parents=True, exist_ok=True)

if not stab_path.exists():
    raise FileNotFoundError(stab_path)

stab = read_vspaero_stab(stab_path)
basic = calculate_stab_basic_indices(stab_path)

mass = 100.0
rho = 1.225
inertia = {
    "Ixx": 1000.0,
    "Iyy": 75.0,
    "Izz": 1000.0,
    "Ixz": 0.0,
}

Uds = 3.0
H = 9.144
gust_start_time = 0.0
gust_end_time = gust_start_time + 2.0 * H / stab.V0
t_final = 3.0

trim_settings = {
    "delta_a": 0.0,
    "delta_e": None,
    "delta_r": 0.0,
    "trim_elevator": True,
    "trim_thrust": False,
    "phi0": 0.0,
    "theta0": None,
    "psi0": 0.0,
    "max_step": 0.02,
}

assert t_final > gust_end_time

display(
    pd.Series(
        {
            "stab_path": str(stab_path),
            "mass": mass,
            "rho_used": rho,
            "rho_in_stab": stab.rho0,
            "Ixx": inertia["Ixx"],
            "Iyy": inertia["Iyy"],
            "Izz": inertia["Izz"],
            "Ixz": inertia["Ixz"],
            "Vinf": stab.V0,
            "Sref": stab.Sref,
            "Bref": stab.Bref,
            "Uds_from_right": Uds,
            "H": H,
            "gust_end_time": gust_end_time,
            "t_final": t_final,
            **trim_settings,
        },
        name="value",
    )
)

### 初期状態に関する注意

本例の初期状態は完全な定常滑空トリムではない。

`trim_elevator=True`により初期ピッチングモーメントを小さくするエレベータ舵角を設定するが、`trim_thrust=False`なので推力はゼロである。また、初期角速度は$p=q=r=0$である。このため、横風突風応答と同時に速度、迎角、ピッチ姿勢などの縦・速度方向の自由応答も含まれる。

## 3. 1-cosine gust波形と外乱横滑り角

突風通過後に速度が0へ戻ることを確認する。

線形モデルではピーク外乱横滑り角を$U_{ds}/V$と近似する。6自由度後処理では$\tan^{-1}(U_{ds}/V)$を使う。

In [ ]:
time = np.linspace(
    0.0,
    1.25 * gust_end_time,
    300,
)
gust_velocity_y = np.array(
    [
        one_cosine_gust_velocity(
            t,
            Uds=Uds,
            H=H,
            V=stab.V0,
            start_time=gust_start_time,
        )
        for t in time
    ]
)

beta_g_peak_linear = Uds / stab.V0
beta_g_peak_6dof = math.atan2(Uds, stab.V0)

display(
    pd.Series(
        {
            "beta_g_peak_linear_rad": beta_g_peak_linear,
            "beta_g_peak_6dof_rad": beta_g_peak_6dof,
            "beta_g_peak_linear_deg": math.degrees(beta_g_peak_linear),
            "beta_g_peak_6dof_deg": math.degrees(beta_g_peak_6dof),
            "relative_difference": (
                beta_g_peak_linear / beta_g_peak_6dof - 1.0
            ),
        },
        name="value",
    )
)

fig, ax = plt.subplots(
    figsize=(7, 4),
    constrained_layout=True,
)
ax.plot(time, gust_velocity_y)
ax.axvline(
    gust_end_time,
    linestyle="--",
    label="gust end",
)
ax.set_xlabel("time [s]")
ax.set_ylabel("gust speed from right [m/s]")
ax.grid(True)
ax.legend()
plt.show()

## 4. 線形4状態モデルとTaylor係数

状態量を

$$
\mathbf{x}
=
\begin{bmatrix}
\beta & \hat p & \hat r & \phi
\end{bmatrix}^{\mathrm T}
$$

とし、

$$
\mathbf{x}'
=
\mathbf{A}\mathbf{x}
+
\mathbf{B}_g\beta_g
$$

を作る。`beta`ではなく`beta_body`に対応する状態量を使う線形モデルである。

In [ ]:
V = float(stab.V0)
Sref = float(stab.Sref)
Bref = float(stab.Bref)
Ixx = inertia["Ixx"]
Izz = inertia["Izz"]
Ixz = inertia["Ixz"]
inertia_det = Ixx * Izz - Ixz**2
qbar = 0.5 * rho * V**2

mu_y = qbar * Sref * Bref / (2.0 * mass * V**2)
mu_inertia = (
    qbar
    * Sref
    * Bref**3
    / (4.0 * V**2 * inertia_det)
)
lambda_g = 9.80665 * Bref / (2.0 * V**2)
alpha0 = float(stab.alpha0)

CY_beta = float(basic["CY_beta"])
CY_phat = float(basic["CY_p"])
CY_rhat = float(basic["CY_r"])
Cl_beta = float(basic["Cl_beta"])
Cl_phat = float(basic["Cl_p"])
Cl_rhat = float(basic["Cl_r"])
Cn_beta = float(basic["Cn_beta"])
Cn_phat = float(basic["Cn_p"])
Cn_rhat = float(basic["Cn_r"])

A = np.array(
    [
        [
            mu_y * CY_beta,
            mu_y * CY_phat + alpha0,
            mu_y * CY_rhat - 1.0,
            lambda_g,
        ],
        [
            mu_inertia * (Izz * Cl_beta + Ixz * Cn_beta),
            mu_inertia * (Izz * Cl_phat + Ixz * Cn_phat),
            mu_inertia * (Izz * Cl_rhat + Ixz * Cn_rhat),
            0.0,
        ],
        [
            mu_inertia * (Ixz * Cl_beta + Ixx * Cn_beta),
            mu_inertia * (Ixz * Cl_phat + Ixx * Cn_phat),
            mu_inertia * (Ixz * Cl_rhat + Ixx * Cn_rhat),
            0.0,
        ],
        [0.0, 1.0, 0.0, 0.0],
    ],
    dtype=float,
)
B_g = np.array(
    [
        mu_y * CY_beta,
        mu_inertia * (Izz * Cl_beta + Ixz * Cn_beta),
        mu_inertia * (Ixz * Cl_beta + Ixx * Cn_beta),
        0.0,
    ],
    dtype=float,
)

A_B_g = A @ B_g
A2_B_g = A @ A_B_g
A3_B_g = A @ A2_B_g

K1 = float(B_g[1])
K2 = float(A_B_g[1])
K3 = float(A2_B_g[1])

# phi' = phat なので、e_phi^T A^n B_g と
# e_p^T A^(n-1) B_g は同じ係数になる。
assert math.isclose(K1, float(A_B_g[3]), rel_tol=1e-12, abs_tol=1e-14)
assert math.isclose(K2, float(A2_B_g[3]), rel_tol=1e-12, abs_tol=1e-14)
assert math.isclose(K3, float(A3_B_g[3]), rel_tol=1e-12, abs_tol=1e-14)

display(pd.DataFrame(A, columns=["beta", "phat", "rhat", "phi"], index=["beta'", "phat'", "rhat'", "phi'"]))
display(pd.Series(B_g, index=["beta'", "phat'", "rhat'", "phi'"], name="B_g"))
display(pd.Series({"K1": K1, "K2": K2, "K3": K3}, name="value"))

## 5. 簡易横風突風ロール指数

定式化記事の簡易指数

$$
K_{\mathrm{gust,roll}}
=
\frac{8I_z}{\rho S b^3}C_{l\beta}
+
C_{l\hat r}C_{n\beta}
$$

を`.stab`から計算する。共通係数を含む主要経路式から同じ値が得られることも確認する。

In [ ]:
mu_x = rho * Sref * Bref**3 / (8.0 * Ixx)
mu_z = rho * Sref * Bref**3 / (8.0 * Izz)

crosswind_gust_roll_index = (
    8.0 * Izz / (rho * Sref * Bref**3) * Cl_beta
    + Cl_rhat * Cn_beta
)

crosswind_gust_path = (
    mu_x * Cl_beta
    + mu_x * mu_z * Cl_rhat * Cn_beta
)
crosswind_gust_roll_index_from_path = (
    crosswind_gust_path / (mu_x * mu_z)
)

assert math.isclose(
    crosswind_gust_roll_index,
    crosswind_gust_roll_index_from_path,
    rel_tol=1e-12,
    abs_tol=1e-14,
)

display(
    pd.Series(
        {
            "Cl_beta": Cl_beta,
            "Cl_rhat": Cl_rhat,
            "Cn_beta": Cn_beta,
            "crosswind_gust_roll_index": crosswind_gust_roll_index,
            "crosswind_gust_roll_index_abs": abs(crosswind_gust_roll_index),
            "crosswind_gust_roll_index_from_path": (
                crosswind_gust_roll_index_from_path
            ),
        },
        name="value",
    )
)

## 6. 横風突風6自由度応答の計算

突風速度は空力相対風へだけ加え、機体の剛体速度状態そのものへ直接加えない。

In [ ]:
history = simulate_6dof_crosswind_gust_from_stab(
    stab_path,
    mass=mass,
    Ixx=inertia["Ixx"],
    Iyy=inertia["Iyy"],
    Izz=inertia["Izz"],
    Ixz=inertia["Ixz"],
    Uds=Uds,
    H=H,
    t_final=t_final,
    rho=rho,
    gust_start_time=gust_start_time,
    **trim_settings,
)

sign_conventions = set(
    history["crosswind_sign_convention"].dropna().astype(str)
)
assert sign_conventions == {"positive_from_right"}
assert history["gust_velocity_y"].max() > 0.0
assert history["airmass_velocity_y"].min() < 0.0
assert history["beta_g"].max() > 0.0

beta_small_disturbance_error = (
    history["beta"]
    - history["beta_body"]
    - history["beta_g"]
)

display(
    history[
        [
            "time",
            "gust_velocity_y",
            "airmass_velocity_y",
            "beta_body",
            "beta_g",
            "beta",
            "p",
            "r",
            "phi",
        ]
    ].head()
)
print("samples:", len(history))
print(
    "max |beta - beta_body - beta_g| [deg]:",
    math.degrees(beta_small_disturbance_error.abs().max()),
)

## 7. 横風突風応答指標

`crosswind_gust_max_abs_beta`は`beta_body`ではなく、有効対気横滑り角`beta`、すなわち$\beta_a$の最大絶対値である。

応答指標の正規化には$\tan^{-1}(U_{ds}/V_{\mathrm{ref}})$を用いる。

In [ ]:
gust_response_indices = calculate_crosswind_gust_response_indices(
    history,
    Vref=stab.V0,
    Bref=stab.Bref,
)

assert math.isclose(
    gust_response_indices["crosswind_gust_beta_peak_input"],
    beta_g_peak_6dof,
    rel_tol=1e-12,
    abs_tol=1e-14,
)

display(
    pd.Series(
        {
            "success": gust_response_indices["crosswind_gust_success"],
            "Uds_from_right": gust_response_indices["crosswind_gust_Uds"],
            "H": gust_response_indices["crosswind_gust_H"],
            "beta_g_peak_input_deg": math.degrees(
                gust_response_indices["crosswind_gust_beta_peak_input"]
            ),
            "max_abs_phi_delta_deg": math.degrees(
                gust_response_indices["crosswind_gust_max_abs_phi_delta"]
            ),
            "max_abs_phi_delta_time": gust_response_indices[
                "crosswind_gust_max_abs_phi_delta_time"
            ],
            "max_abs_beta_air_deg": math.degrees(
                gust_response_indices["crosswind_gust_max_abs_beta"]
            ),
            "max_abs_phat": gust_response_indices[
                "crosswind_gust_max_abs_phat"
            ],
            "max_abs_rhat": gust_response_indices[
                "crosswind_gust_max_abs_rhat"
            ],
            "bank_index_abs_per_beta_g": gust_response_indices[
                "crosswind_gust_bank_index_abs_per_beta_g"
            ],
            "simple_crosswind_gust_roll_index": (
                crosswind_gust_roll_index
            ),
            "message": gust_response_indices[
                "crosswind_gust_message"
            ],
        },
        name="value",
    )
)

## 8. 時刻歴の保存と表示

In [ ]:
csv_path = write_6dof_history_csv(
    history,
    output_dir / "crosswind_gust_6dof_history.csv",
)

plot_path = output_dir / "crosswind_gust_6dof_history.png"
fig = plot_6dof_history(
    history,
    plot_path=plot_path,
    show=True,
    degrees=True,
)

print("csv_path:", csv_path)
print("plot_path:", plot_path)

## 9. 確認事項

- `crosswind_sign_convention`が`positive_from_right`
- `Uds>0`のとき`gust_velocity_y>0`
- `Uds>0`のとき`airmass_velocity_y<0`
- `Uds>0`のとき`beta_g>0`
- 線形状態量$\beta$は`beta_body`、有効対気横滑り角$\beta_a$は`beta`
- 突風通過後に`gust_velocity_y=0`
- $K_1,K_2,K_3$の二つの行列表現が一致
- `crosswind_gust_roll_index`が主要経路式から得た値と一致
- 質量・慣性モーメント・密度と`.stab`が同じ単位系
- 線形微係数の適用範囲を超える大きな横滑り角になっていないか
- 初期状態は完全な定常滑空トリムではなく、縦・速度方向の自由応答を含む